# LLM As Judge

This notebook evaluates generated NLG reports with OpenAI `gpt-5` on a 1-5 scale for:

- `No-Omissions`
- `No-Additions`
- `Grammaticality`
- `Coherence`
- `Fluency`

It pairs each generated report with its source input bundle and, when available, includes the immediately previous generated report as continuity context.

The judge output matches this schema:

```json
{
  "No-Omissions": {"Justification": "", "Score": ""},
  "No-Additions": {"Justification": "", "Score": ""},
  "Grammaticality": {"Justification": "", "Score": ""},
  "Coherence": {"Justification": "", "Score": ""},
  "Fluency": {"Justification": "", "Score": ""}
}
```


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import os
import re
import sys
from typing import Any, Optional

import pandas as pd
from IPython.display import JSON, display
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "load_data.py").exists():
    fallback = Path("/home/chinonso/PHD_PROJECTS/Financial-D2T-Agent")
    if fallback.exists():
        PROJECT_ROOT = fallback

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from load_data import build_multi_stock_prompt_context, load_generation_samples

PROJECT_ROOT


PosixPath('/home/chinonso/PHD_PROJECTS/Financial-D2T-Agent')

## Run LLM-AS-JUDGE

In [2]:
REGION = "us"
SOURCE_MODEL = "gpt-5"
SOURCE_ARCH = "workflow"
SOURCE_REFLECTION = True

NLG_PROVIDER = "openai"
NLG_MODEL = "gpt-5"
LANGUAGE = "en"
WORKFLOW = "no_guardrail_no_finalizer"  # "default" or "e2e" or "default_old"
    # (True, "no_orchestrator_no_guardrail_no_finalizer"),
    # (True, "no_orchestrator_no_finalizer"),
    # (True, "no_guardrail_no_finalizer"),

JUDGE_MODEL = "gpt-5"
JUDGE_REASONING_EFFORT = "high"
JUDGE_MAX_OUTPUT_TOKENS = 16000
JUDGE_MAX_RETRIES = 3

LIMIT = None
OVERWRITE = False

In [3]:
def current_source_dataset_dir() -> Path:
    return (
        PROJECT_ROOT
        / "results"
        / f"final_report2025_{REGION}"
        / SOURCE_MODEL
        / f"{SOURCE_ARCH}_{SOURCE_REFLECTION}"
    )


def current_nlg_output_dir() -> Path:
    return (
        PROJECT_ROOT
        / "results"
        / "nlg"
        / f"final_report2025_{REGION}"
        / SOURCE_MODEL
        / f"{SOURCE_ARCH}_{SOURCE_REFLECTION}"
        / NLG_PROVIDER
        / NLG_MODEL
        / LANGUAGE
        / WORKFLOW
    )


def current_judge_output_dir() -> Path:
    return (
        PROJECT_ROOT
        / "results"
        / "validation"
        / "llm_judge"
        / REGION
        / SOURCE_MODEL
        / f"{SOURCE_ARCH}_{SOURCE_REFLECTION}"
        / NLG_PROVIDER
        / NLG_MODEL
        / LANGUAGE
        / WORKFLOW
    )


def print_current_paths() -> None:
    print("Source dataset:", current_source_dataset_dir())
    print("Generated reports:", current_nlg_output_dir())
    print("Judge outputs:", current_judge_output_dir())
    print("Judge model:", JUDGE_MODEL)


print_current_paths()


Source dataset: /home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/final_report2025_us/gpt-5/workflow_True
Generated reports: /home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/no_guardrail_no_finalizer
Judge outputs: /home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/validation/llm_judge/us/gpt-5/workflow_True/openai/gpt-5/en/no_guardrail_no_finalizer
Judge model: gpt-5


In [4]:
JUDGE_INSTRUCTIONS = """You are evaluating how well a Generated Report realises a given Input Bundle for a financial data-to-text task.

Your task:
1. Read the Input Bundle and the Generated Report carefully.
2. For each of the five Dimensions below, assign a score from 1 (lowest) to 5 (highest).
3. For each Dimension, give a short justification (one or two sentences).
4. Return only a single JSON object in the exact format specified. Do not include any extra text.

Dimensions:
No-Omissions: To what degree is ALL the information in the Input Bundle present in the Generated Report. Judge only against the structured data fields in the Input Bundle. Do not penalise the absence of year-over-year comparisons, prior-period figures, or growth rates that appear only in recommendation justification text and not in the structured data fields.
No-Additions: To what degree does the Generated Report include ONLY information from the Input Bundle. The generation-prompt report metadata supplied at the top of the Input Bundle is always part of the input and must never be penalised as an addition. This includes inaugural coverage status, which is determined by whether a previous report is present in the input. Qualitative inferences directly derivable from the figures in the Input Bundle, including risk statements, catalyst statements, and analytical characterisations grounded in the supplied metrics, are permitted. Only penalise specific numeric figures or factual claims that cannot be found in or derived from the Input Bundle by simple arithmetic. Do not use outside knowledge about the companies.
Grammaticality: To what degree is the Generated Report free of grammatical errors, considering form only.
Coherence: To what degree is the Generated Report well structured and organised into a coherent body of information about the covered stocks, from the perspective of meaning only.
Fluency: To what degree does the Generated Report read smoothly and naturally as professional financial prose, without abrupt or awkward phrasing.

Important notes:
- No-Omissions and No-Additions are judged only with respect to the Input Bundle. Do not use outside knowledge about the companies, markets, or financial figures.
- Grammaticality, Coherence, and Fluency are intrinsic properties of the Generated Report. You do not need the Input Bundle to judge them.
- Scores must be integers in the set (1, 2, 3, 4, 5).
- Judge each Dimension independently.
- Do not award 5 by default unless the condition for that score is fully satisfied.

Return your assessment in this exact JSON format, with no additional keys and no extra text:
{{
  "No-Omissions": {{"Justification": "", "Score": ""}},
  "No-Additions": {{"Justification": "", "Score": ""}},
  "Grammaticality": {{"Justification": "", "Score": ""}},
  "Coherence": {{"Justification": "", "Score": ""}},
  "Fluency": {{"Justification": "", "Score": ""}}
}}

Input Bundle:
{input_bundle}

Previous Report (continuity context only, may be absent):
{previous_report}

Generated Report:
{generated_report}
"""

In [5]:
DIMENSION_NAMES = [
    "No-Omissions",
    "No-Additions",
    "Grammaticality",
    "Coherence",
    "Fluency",
]


class DimensionScore(BaseModel):
    Justification: str = Field(min_length=1)
    Score: int = Field(ge=1, le=5)


class JudgeScorecard(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    no_omissions: DimensionScore = Field(alias="No-Omissions")
    no_additions: DimensionScore = Field(alias="No-Additions")
    grammaticality: DimensionScore = Field(alias="Grammaticality")
    coherence: DimensionScore = Field(alias="Coherence")
    fluency: DimensionScore = Field(alias="Fluency")


def safe_slug(text: str) -> str:
    value = (text or "sample").strip()
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value)


def extract_generated_text(payload: dict[str, Any], txt_path: Optional[Path] = None) -> str:
    for key in ("generated_text", "final_response", "report"):
        value = payload.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()

    raw_output = payload.get("raw_output")
    if isinstance(raw_output, dict):
        for key in ("output_text", "text"):
            value = raw_output.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip()

    if txt_path and txt_path.is_file():
        return txt_path.read_text(encoding="utf-8").strip()

    raise ValueError("Could not find generated text in payload or matching .txt file.")


def load_generated_results(output_dir: Path) -> list[dict[str, Any]]:
    if not output_dir.exists():
        raise FileNotFoundError(f"Generated output directory not found: {output_dir}")

    rows: list[dict[str, Any]] = []
    for json_path in sorted(output_dir.glob("*.json")):
        if json_path.name.endswith("_sequence_summary.json"):
            continue

        payload = json.loads(json_path.read_text(encoding="utf-8"))
        txt_path = json_path.with_suffix(".txt")
        sample_meta = payload.get("sample_metadata", {}) or {}
        sample_name = sample_meta.get("sample_name") or payload.get("sample_name") or json_path.stem
        analysis_date = sample_meta.get("analysis_date") or payload.get("analysis_date")
        generated_text = extract_generated_text(payload, txt_path=txt_path)

        rows.append(
            {
                "sample_name": str(sample_name),
                "analysis_date": str(analysis_date or "")[:10],
                "generated_text": generated_text,
                "json_path": json_path,
                "txt_path": txt_path if txt_path.exists() else None,
                "payload": payload,
                "token_usage": payload.get("token_usage", {}),
            }
        )

    rows.sort(key=lambda row: (row["analysis_date"], row["sample_name"]))
    return rows


def build_sample_index(dataset_path: Path) -> dict[str, dict[str, Any]]:
    if not dataset_path.exists():
        raise FileNotFoundError(f"Source dataset directory not found: {dataset_path}")

    samples = load_generation_samples(
        dataset_path=str(dataset_path),
        dataset_kind="auto",
        min_stocks_per_month=1,
    )
    return {str(sample["sample_name"]): sample for sample in samples}


def build_effective_prompt_context(sample: dict[str, Any], previous_generated_text: str) -> str:
    if sample.get("sample_type") == "multi_stock_monthly":
        return build_multi_stock_prompt_context(
            analysis_date=str(sample.get("analysis_date", "")),
            stock_rows=sample.get("stocks", []),
            previous_report=previous_generated_text or "N/A",
        )

    prompt_context = sample.get("prompt_context")
    if isinstance(prompt_context, str) and prompt_context.strip():
        return prompt_context

    return json.dumps(sample.get("data_input", ""), ensure_ascii=False, indent=2)


def infer_coverage_end_date(sample_index: dict[str, dict[str, Any]]) -> str:
    candidate_dates = [
        str(sample.get("analysis_date", "")).strip()[:10]
        for sample in sample_index.values()
        if str(sample.get("analysis_date", "")).strip()
    ]
    if not candidate_dates:
        return ""

    latest = max(candidate_dates)
    parsed = pd.to_datetime(latest, errors="coerce")
    if pd.isna(parsed):
        return latest
    return (parsed + pd.offsets.MonthEnd(0)).date().isoformat()


def build_report_metadata(sample: dict[str, Any], coverage_end_date: str) -> dict[str, str]:
    analysis_date = str(sample.get("analysis_date", "")).strip()[:10]
    tickers_value = sample.get("tickers") or []
    tickers_text = ", ".join(str(t) for t in tickers_value) if isinstance(tickers_value, list) else str(tickers_value or "")

    ticker_count = sample.get("ticker_count")
    if ticker_count in (None, "") and isinstance(tickers_value, list):
        ticker_count = len(tickers_value)

    horizon_months = ""
    analysis_ts = pd.to_datetime(analysis_date, errors="coerce")
    end_ts = pd.to_datetime(coverage_end_date, errors="coerce")
    if not pd.isna(analysis_ts) and not pd.isna(end_ts):
        horizon_months = str(max((end_ts.year - analysis_ts.year) * 12 + (end_ts.month - analysis_ts.month), 0))

    return {
        "analysis_date": analysis_date,
        "tickers": tickers_text,
        "ticker_count": str(ticker_count or ""),
        "end_date": coverage_end_date,
        "horizon_months": horizon_months,
    }


def build_evaluation_records(
    sample_index: dict[str, dict[str, Any]],
    generated_rows: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    generated_by_date = {
        row["analysis_date"]: row
        for row in generated_rows
        if row.get("analysis_date")
    }
    coverage_end_date = infer_coverage_end_date(sample_index)

    records: list[dict[str, Any]] = []
    for row in generated_rows:
        sample = sample_index.get(row["sample_name"])
        if sample is None:
            print(f"Skipping {row['sample_name']}: no matching source sample found.")
            continue

        previous_analysis_date = str(sample.get("previous_analysis_date") or "")[:10]
        previous_generated_text = "N/A"

        if previous_analysis_date and previous_analysis_date in generated_by_date:
            previous_generated_text = generated_by_date[previous_analysis_date]["generated_text"]
        else:
            fallback_previous = sample.get("previous_report")
            if isinstance(fallback_previous, str) and fallback_previous.strip():
                previous_generated_text = fallback_previous.strip()

        prompt_context = build_effective_prompt_context(sample, previous_generated_text)
        report_metadata = build_report_metadata(sample, coverage_end_date)

        records.append(
            {
                "sample_name": row["sample_name"],
                "analysis_date": row["analysis_date"],
                "previous_analysis_date": previous_analysis_date or None,
                "generated_text": row["generated_text"],
                "previous_generated_text": previous_generated_text,
                "prompt_context": prompt_context,
                "report_metadata": report_metadata,
                "sample": sample,
                "json_path": row["json_path"],
                "txt_path": row["txt_path"],
                "token_usage": row["token_usage"],
            }
        )

    records.sort(key=lambda record: (record["analysis_date"], record["sample_name"]))
    return records


def build_records_from_current_config() -> list[dict[str, Any]]:
    source_dataset_dir = current_source_dataset_dir()
    nlg_output_dir = current_nlg_output_dir()
    sample_index = build_sample_index(source_dataset_dir)
    generated_rows = load_generated_results(nlg_output_dir)
    return build_evaluation_records(sample_index, generated_rows)


def build_judge_input(record: dict[str, Any]) -> str:
    report_metadata = record.get("report_metadata", {}) or {}
    return f"""Sample name: {record['sample_name']}
Analysis date: {record['analysis_date']}
Previous analysis date: {record.get('previous_analysis_date') or 'N/A'}

Generation-prompt report metadata (treat these as authoritative input facts, not additions):
Analysis month for this report: {report_metadata.get('analysis_date', '')}
Current-month coverage: {report_metadata.get('tickers', '')}
Number of stocks in bundle: {report_metadata.get('ticker_count', '')}
Coverage window end date: {report_metadata.get('end_date', '')}
Investment horizon for this report: {report_metadata.get('horizon_months', '')} months

Current-month input bundle:
{record['prompt_context']}

Previous generated report for continuity context only:
{record['previous_generated_text']}

Generated report to score:
{record['generated_text']}"""


def response_usage_dict(response: Any) -> dict[str, Any]:
    usage = getattr(response, "usage", None)
    if usage is None:
        return {}
    if hasattr(usage, "to_dict"):
        return usage.to_dict()
    if hasattr(usage, "model_dump"):
        return usage.model_dump()
    if isinstance(usage, dict):
        return usage

    data: dict[str, Any] = {}
    for name in dir(usage):
        if name.startswith("_"):
            continue
        value = getattr(usage, name)
        if callable(value):
            continue
        if isinstance(value, (str, int, float, bool, dict, list, type(None))):
            data[name] = value
    return data


def compact_debug_value(value: Any) -> Any:
    if value is None:
        return None
    if hasattr(value, "model_dump"):
        return value.model_dump()
    if hasattr(value, "to_dict"):
        return value.to_dict()
    if isinstance(value, (dict, list, str, int, float, bool)):
        return value
    return str(value)


def extract_response_text(response: Any) -> str:
    output_text = getattr(response, "output_text", None)
    if isinstance(output_text, str) and output_text.strip():
        return output_text.strip()

    segments: list[str] = []
    for output in getattr(response, "output", []) or []:
        if getattr(output, "type", None) != "message":
            continue
        for content in getattr(output, "content", []) or []:
            content_type = getattr(content, "type", None)
            if content_type == "output_text":
                text = getattr(content, "text", None)
                if isinstance(text, str) and text.strip():
                    segments.append(text.strip())
            elif content_type == "refusal":
                refusal = getattr(content, "refusal", None)
                if isinstance(refusal, str) and refusal.strip():
                    segments.append(refusal.strip())

    return "\n".join(segments).strip()


def parse_scorecard_from_text(raw_text: str) -> JudgeScorecard:
    text = (raw_text or "").strip()
    if not text:
        raise ValueError("Judge response text is empty.")

    candidates = [text]
    if text.startswith("```"):
        fenced = re.sub(r"^```(?:json)?\s*", "", text)
        fenced = re.sub(r"\s*```$", "", fenced)
        candidates.append(fenced.strip())

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        candidates.append(text[start:end + 1].strip())

    seen: set[str] = set()
    last_error: Exception | None = None
    for candidate in candidates:
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)
        try:
            return JudgeScorecard.model_validate_json(candidate)
        except Exception as exc:
            last_error = exc

    raise ValueError(f"Could not parse judge response text into JudgeScorecard: {last_error}")


def judge_one(record: dict[str, Any], client: OpenAI) -> dict[str, Any]:
    last_error: Exception | None = None

    for attempt in range(1, JUDGE_MAX_RETRIES + 1):
        try:
            response = client.responses.parse(
                model=JUDGE_MODEL,
                instructions=JUDGE_INSTRUCTIONS,
                input=build_judge_input(record),
                text_format=JudgeScorecard,
                reasoning={"effort": JUDGE_REASONING_EFFORT},
                # max_output_tokens = JUDGE_MAX_OUTPUT_TOKENS,
                text={"verbosity": "low"},
                store=False,
            )

            parsed = response.output_parsed
            if parsed is None:
                raw_text = extract_response_text(response)
                if raw_text:
                    parsed = parse_scorecard_from_text(raw_text)
                else:
                    raise ValueError(
                        f"No parsed judge output returned for {record['sample_name']}. "
                        f"status={getattr(response, 'status', None)!r}; "
                        f"incomplete_details={compact_debug_value(getattr(response, 'incomplete_details', None))!r}"
                    )

            scorecard = parsed.model_dump(by_alias=True)
            return {
                "sample_name": record["sample_name"],
                "analysis_date": record["analysis_date"],
                "previous_analysis_date": record.get("previous_analysis_date"),
                "judge_model": JUDGE_MODEL,
                "judge_reasoning_effort": JUDGE_REASONING_EFFORT,
                "judge_attempt": attempt,
                "source_json_path": str(record["json_path"]),
                "source_txt_path": str(record["txt_path"]) if record["txt_path"] else None,
                # "source_token_usage": record.get("token_usage", {}),
                "judge_usage": response_usage_dict(response),
                "scores": scorecard,
            }
        except Exception as exc:
            last_error = exc
            print(
                f"[judge retry {attempt}/{JUDGE_MAX_RETRIES}] "
                f"{record['sample_name']}: {type(exc).__name__}: {exc}"
            )

    raise RuntimeError(
        f"Judging failed for {record['sample_name']} after {JUDGE_MAX_RETRIES} attempt(s)."
    ) from last_error


def flatten_scorecard(judgement: dict[str, Any]) -> dict[str, Any]:
    row = {
        "sample_name": judgement["sample_name"],
        "analysis_date": judgement["analysis_date"],
        "previous_analysis_date": judgement.get("previous_analysis_date"),
        "judge_model": judgement["judge_model"],
        "judge_reasoning_effort": judgement["judge_reasoning_effort"],
    }
    numeric_scores: list[int] = []

    for dimension in DIMENSION_NAMES:
        key = dimension.lower().replace("-", "_")
        value = judgement["scores"][dimension]
        row[f"{key}_score"] = int(value["Score"])
        row[f"{key}_justification"] = value["Justification"]
        numeric_scores.append(int(value["Score"]))

    row["mean_score"] = sum(numeric_scores) / len(numeric_scores)
    return row


def infer_judge_output_dir(records: list[dict[str, Any]]) -> Path:
    if not records:
        raise ValueError("Cannot infer judge output directory from an empty record set.")

    nlg_root = (PROJECT_ROOT / "results" / "nlg").resolve()
    first_parent = Path(records[0]["json_path"]).resolve().parent
    try:
        relative = first_parent.relative_to(nlg_root)
    except ValueError as exc:
        raise ValueError(f"Record path is not under the NLG results root: {first_parent}") from exc

    parts = relative.parts
    if len(parts) < 7:
        raise ValueError(f"Unexpected NLG result path layout: {first_parent}")

    dataset_part = parts[0]
    region_prefix = "final_report2025_"
    if not dataset_part.startswith(region_prefix):
        raise ValueError(f"Could not infer region from dataset folder: {dataset_part}")

    region = dataset_part[len(region_prefix):]
    for record in records[1:]:
        candidate_parent = Path(record["json_path"]).resolve().parent
        try:
            candidate_relative = candidate_parent.relative_to(nlg_root)
        except ValueError as exc:
            raise ValueError(f"Record path is not under the NLG results root: {candidate_parent}") from exc
        if candidate_relative.parts != parts:
            raise ValueError(
                "Records do not all belong to the same NLG output directory. "
                f"First={first_parent} Candidate={candidate_parent}"
            )

    inferred = (
        PROJECT_ROOT
        / "results"
        / "validation"
        / "llm_judge"
        / region
        / parts[1]
        / parts[2]
        / parts[3]
        / parts[4]
        / parts[5]
        / parts[6]
    )
    return inferred


def validate_records_match_current_config(records: list[dict[str, Any]]) -> None:
    if not records:
        return
    expected_nlg_dir = current_nlg_output_dir().resolve()
    actual_nlg_dir = Path(records[0]["json_path"]).resolve().parent
    if actual_nlg_dir != expected_nlg_dir:
        raise ValueError(
            "Provided records do not match the current notebook config. "
            f"Expected NLG directory={expected_nlg_dir} but records came from {actual_nlg_dir}. "
            "Rerun the config cell and then call run_judging() with no argument."
        )


def run_judging(records: Optional[list[dict[str, Any]]] = None) -> tuple[list[dict[str, Any]], pd.DataFrame]:
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set in the environment."
    source_dataset_dir = current_source_dataset_dir()
    nlg_output_dir = current_nlg_output_dir()
    config_judge_output_dir = current_judge_output_dir()
    print("[judge config] Source dataset:", source_dataset_dir)
    print("[judge config] Generated reports:", nlg_output_dir)
    print("[judge config] Judge outputs:", config_judge_output_dir)
    if records is None:
        records = build_records_from_current_config()
    else:
        validate_records_match_current_config(records)

    if not records:
        raise ValueError("No evaluation records were built. Check your source dataset and NLG output directory settings.")

    judge_output_dir = infer_judge_output_dir(records)
    if str(judge_output_dir) != str(config_judge_output_dir):
        print(f"[judge output override] Using inferred directory from records: {judge_output_dir}")
        print(f"[judge output override] Current config judge directory was: {config_judge_output_dir}")
    judge_output_dir.mkdir(parents=True, exist_ok=True)
    client = OpenAI()

    selected = records if LIMIT is None else records[:LIMIT]
    if not selected:
        raise ValueError("No records selected for judging. Check LIMIT and the matched report set.")

    collected: list[dict[str, Any]] = []

    for idx, record in enumerate(selected, start=1):
        out_path = judge_output_dir / f"{safe_slug(record['sample_name'])}.json"

        if out_path.exists() and not OVERWRITE:
            saved = json.loads(out_path.read_text(encoding="utf-8"))
            collected.append(saved)
            print(f"[skip {idx}/{len(selected)}] {record['sample_name']} -> {out_path.name}")
            continue

        try:
            judgement = judge_one(record, client)
        except Exception as exc:
            error_payload = {
                "sample_name": record["sample_name"],
                "analysis_date": record["analysis_date"],
                "judge_model": JUDGE_MODEL,
                "judge_reasoning_effort": JUDGE_REASONING_EFFORT,
                "source_json_path": str(record["json_path"]),
                "source_txt_path": str(record["txt_path"]) if record["txt_path"] else None,
                "error": f"{type(exc).__name__}: {exc}",
            }
            out_path.write_text(json.dumps(error_payload, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"[error {idx}/{len(selected)}] {record['sample_name']} -> {out_path.name}")
            continue

        out_path.write_text(json.dumps(judgement, ensure_ascii=False, indent=2), encoding="utf-8")
        collected.append(judgement)
        print(f"[done {idx}/{len(selected)}] {record['sample_name']} -> {out_path.name}")

    flattened = [flatten_scorecard(item) for item in collected if item.get("scores")]
    if flattened:
        summary = pd.DataFrame(flattened).sort_values(["analysis_date", "sample_name"])
    else:
        summary = pd.DataFrame(
            columns=[
                "sample_name",
                "analysis_date",
                "previous_analysis_date",
                "judge_model",
                "judge_reasoning_effort",
                "mean_score",
            ]
        )
    summary_path = judge_output_dir / "summary.csv"
    summary.to_csv(summary_path, index=False)

    bundle_path = judge_output_dir / "all_judgements.json"
    bundle_path.write_text(json.dumps(collected, ensure_ascii=False, indent=2), encoding="utf-8")

    print("Saved summary:", summary_path)
    print("Saved bundle:", bundle_path)
    return collected, summary


In [6]:
sample_index = build_sample_index(current_source_dataset_dir())
generated_rows = load_generated_results(current_nlg_output_dir())
records = build_evaluation_records(sample_index, generated_rows)

overview = pd.DataFrame(
    {
        "sample_name": [record["sample_name"] for record in records],
        "analysis_date": [record["analysis_date"] for record in records],
        "previous_analysis_date": [record["previous_analysis_date"] for record in records],
        "source_json_path": [str(record["json_path"]) for record in records],
    }
)

# display(overview)

In [7]:
num = 1
till = 1000
if records:
    preview = {
        "sample_name": records[num]["sample_name"],
        "analysis_date": records[num]["analysis_date"],
        "previous_analysis_date": records[num]["previous_analysis_date"],
        "prompt_context_preview": records[num]["prompt_context"][:till],
        "previous_generated_text_preview": records[num]["previous_generated_text"][:till],
        "generated_text_preview": records[num]["generated_text"][:till],
    }
    # display(JSON(preview, expanded=False))


In [8]:
judgements, judge_summary = run_judging()
display(judge_summary)

score_columns = [column for column in judge_summary.columns if column.endswith("_score")]
display(judge_summary[score_columns].mean().to_frame(name="mean_score").T)


[judge config] Source dataset: /home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/final_report2025_us/gpt-5/workflow_True
[judge config] Generated reports: /home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/nlg/final_report2025_us/gpt-5/workflow_True/openai/gpt-5/en/no_guardrail_no_finalizer
[judge config] Judge outputs: /home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/validation/llm_judge/us/gpt-5/workflow_True/openai/gpt-5/en/no_guardrail_no_finalizer


[done 1/14] multi_stock_2025-01-31 -> multi_stock_2025-01-31.json
[done 2/14] multi_stock_2025-02-28 -> multi_stock_2025-02-28.json
[done 3/14] multi_stock_2025-03-31 -> multi_stock_2025-03-31.json
[done 4/14] multi_stock_2025-04-30 -> multi_stock_2025-04-30.json
[done 5/14] multi_stock_2025-05-31 -> multi_stock_2025-05-31.json
[done 6/14] multi_stock_2025-06-30 -> multi_stock_2025-06-30.json
[done 7/14] multi_stock_2025-07-31 -> multi_stock_2025-07-31.json
[done 8/14] multi_stock_2025-08-31 -> multi_stock_2025-08-31.json
[done 9/14] multi_stock_2025-09-30 -> multi_stock_2025-09-30.json
[done 10/14] multi_stock_2025-10-31 -> multi_stock_2025-10-31.json
[done 11/14] multi_stock_2025-11-30 -> multi_stock_2025-11-30.json
[done 12/14] multi_stock_2025-12-31 -> multi_stock_2025-12-31.json
[done 13/14] multi_stock_2026-01-31 -> multi_stock_2026-01-31.json
[done 14/14] multi_stock_2026-02-25 -> multi_stock_2026-02-25.json
Saved summary: /home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/

,sample_name,analysis_date,previous_analysis_date,judge_model,judge_reasoning_effort,no_omissions_score,no_omissions_justification,no_additions_score,no_additions_justification,grammaticality_score,grammaticality_justification,coherence_score,coherence_justification,fluency_score,fluency_justification,mean_score
0,multi_stock_2025-01-31,2025-01-31,None,gpt-5,high,5,All 8 tickers are covered with correct recomme...,5,All figures and claims come from the bundle or...,5,The report is free of grammatical errors and u...,5,Well organized from summary and methodology th...,5,Reads smoothly as professional financial prose...,5.0
1,multi_stock_2025-02-28,2025-02-28,2025-01-31,gpt-5,high,4,"All eight tickers include the correct ratings,...",5,All figures are from the bundle or directly fr...,5,Text is free of grammatical errors and well pu...,5,Report is logically organized with clear secti...,5,Reads smoothly in a professional financial sty...,4.8
2,multi_stock_2025-03-31,2025-03-31,2025-02-28,gpt-5,high,4,"Includes the correct recommendations, targets,...",5,All quantitative claims are sourced from the b...,5,Grammar and punctuation are correct throughout...,5,Information is clearly structured from executi...,5,"Prose is concise and professional, reading smo...",4.8
3,multi_stock_2025-04-30,2025-04-30,2025-03-31,gpt-5,high,4,Covers all tickers with correct recommendation...,5,All numeric claims are sourced from or directl...,5,No grammatical errors noted; sentence structur...,5,Well-organized from executive summary to per-t...,5,Reads smoothly with professional financial ton...,4.8
4,multi_stock_2025-05-31,2025-05-31,2025-04-30,gpt-5,high,5,All eight tickers are covered with recommendat...,5,All figures are drawn from the structured inpu...,5,The text is free of grammatical errors and mai...,5,Well-organized from executive summary through ...,5,Reads smoothly with professional financial ton...,5.0
5,multi_stock_2025-06-30,2025-06-30,2025-05-31,gpt-5,high,5,"All tickers’ recommendations, targets, current...",5,All numbers come from structured fields or the...,5,The report is free of grammatical errors and m...,5,Information is logically organized from summar...,5,"Professional, concise financial prose with smo...",5.0
6,multi_stock_2025-07-31,2025-07-31,2025-06-30,gpt-5,high,5,All eight tickers are covered with correct rec...,5,All quantitative claims align with the structu...,5,No grammatical errors detected; punctuation an...,5,The report is logically organized from executi...,5,Reads smoothly as professional financial prose...,5.0
7,multi_stock_2025-08-31,2025-08-31,2025-07-31,gpt-5,high,4,All eight tickers have correct recommendations...,5,All figures cited come from the bundle (metric...,5,No clear grammatical errors.,5,Well-structured from summary and methodology t...,4,"Generally smooth professional prose, with only...",4.6
8,multi_stock_2025-09-30,2025-09-30,2025-08-31,gpt-5,high,4,All 8 tickers are covered with correct recomme...,5,All figures trace to the structured fields or ...,5,The report is free of grammatical errors and m...,5,"Well-structured with clear sections (summary, ...",5,Reads smoothly in professional financial prose...,4.8
9,multi_stock_2025-10-31,2025-10-31,2025-09-30,gpt-5,high,5,Includes all report metadata and all 8 tickers...,5,All figures and facts are drawn from the struc...,5,"Clean, error‑free sentences with correct punct...",5,Logically organized with clear sections (summa...,5,Reads smoothly in professional financial prose...,5.0


,no_omissions_score,no_additions_score,grammaticality_score,coherence_score,fluency_score,mean_score
mean_score,4.642857,5.0,5.0,5.0,4.928571,4.914286


## Results

In [9]:
comparison_rows = []
comparison_root = PROJECT_ROOT / "results" / "validation" / "llm_judge" / "us" / "gpt-5"
comparison_variants = [
    (False, "default"),
    (False, "e2e"),
    (True, "default"),
    (True, "e2e"),
    (True, "no_orchestrator_no_guardrail_no_finalizer"),
    (True, "no_orchestrator_no_finalizer"),
    (True, "no_guardrail_no_finalizer"),
    
]
dimension_score_columns = [
    "no_omissions_score",
    "no_additions_score",
    "grammaticality_score",
    "coherence_score",
    "fluency_score",
]

for source_reflection, workflow_name in comparison_variants:
    summary_path = (
        comparison_root
        / f"workflow_{source_reflection}"
        / "openai"
        / "gpt-5"
        / "en"
        / workflow_name
        / "summary.csv"
    )

    if not summary_path.exists():
        print(f"Missing summary: {summary_path}")
        continue

    summary_df = pd.read_csv(summary_path)
    if summary_df.empty:
        print(f"Empty summary: {summary_path}")
        continue

    row = {
        "source_reflection": source_reflection,
        "workflow": workflow_name,
        "n_samples": len(summary_df),
        "summary_path": str(summary_path),
    }
    for column in dimension_score_columns:
        row[column] = summary_df[column].mean()
    row["mean_score"] = summary_df["mean_score"].mean()
    comparison_rows.append(row)

if not comparison_rows:
    raise ValueError("No comparison summaries were found for gpt-5.")

comparison_table = pd.DataFrame(comparison_rows)
comparison_table["reflection_label"] = comparison_table["source_reflection"].map({False: "False", True: "True"})
comparison_table = comparison_table.sort_values(["source_reflection", "workflow"]).reset_index(drop=True)

display_columns = [
    "reflection_label",
    "workflow",
    "n_samples",
    "no_omissions_score",
    "no_additions_score",
    "grammaticality_score",
    "coherence_score",
    "fluency_score",
    "mean_score",
]
comparison_display = comparison_table[display_columns].rename(columns={"reflection_label": "source_reflection"})
display(comparison_display.round(3))

comparison_save_path = comparison_root / "gpt-5_workflow_true_false_default_e2e_comparison.csv"
comparison_table.to_csv(comparison_save_path, index=False)
print("Saved comparison table:", comparison_save_path)


,source_reflection,workflow,n_samples,no_omissions_score,no_additions_score,grammaticality_score,coherence_score,fluency_score,mean_score
0,False,default,14,4.643,5.000,4.929,5.000,5.000,4.914
1,False,e2e,14,4.571,4.857,4.929,5.000,5.000,4.871
2,True,default,14,4.643,5.000,4.929,4.929,5.000,4.900
3,True,e2e,14,4.643,4.857,5.000,4.929,5.000,4.886
4,True,no_guardrail_no_finalizer,14,4.643,5.000,5.000,5.000,4.929,4.914
5,True,no_orchestrator_no_finalizer,14,4.714,4.071,5.000,4.929,5.000,4.743
6,True,no_orchestrator_no_guardrail_no_finalizer,14,4.857,5.000,4.929,5.000,5.000,4.957


Saved comparison table: /home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/validation/llm_judge/us/gpt-5/gpt-5_workflow_true_false_default_e2e_comparison.csv


In [10]:
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path
from IPython.display import display

comparison_root = PROJECT_ROOT / "results" / "validation" / "llm_judge" / "us" / "gpt-5"
comparison_variants = [
    (False, "default"),
    (False, "e2e"),
    (True, "default"),
    (True, "e2e"),
    (True, "no_orchestrator_no_guardrail_no_finalizer"),
    (True, "no_orchestrator_no_finalizer"),
    (True, "no_guardrail_no_finalizer"),
]
dimension_score_columns = [
    "no_omissions_score",
    "no_additions_score",
    "grammaticality_score",
    "coherence_score",
    "fluency_score",
]


def bootstrap_ci(scores, n_bootstrap=10000, ci=0.95):
    """Compute bootstrap confidence interval for the mean of a score array."""
    scores = np.array(scores)
    bootstrap_means = np.array([
        np.random.choice(scores, size=len(scores), replace=True).mean()
        for _ in range(n_bootstrap)
    ])
    alpha = (1 - ci) / 2
    lower = np.percentile(bootstrap_means, alpha * 100)
    upper = np.percentile(bootstrap_means, (1 - alpha) * 100)
    return lower, upper


# ── Step 1: Load per-month scores for every configuration ────────────────────
config_scores = {}   # key: (source_reflection, workflow) -> DataFrame of raw scores
comparison_rows = []

for source_reflection, workflow_name in comparison_variants:
    summary_path = (
        comparison_root
        / f"workflow_{source_reflection}"
        / "openai"
        / "gpt-5"
        / "en"
        / workflow_name
        / "summary.csv"
    )

    if not summary_path.exists():
        print(f"Missing summary: {summary_path}")
        continue

    summary_df = pd.read_csv(summary_path)
    if summary_df.empty:
        print(f"Empty summary: {summary_path}")
        continue

    config_scores[(source_reflection, workflow_name)] = summary_df
    n = len(summary_df)

    row = {
        "source_reflection": source_reflection,
        "workflow": workflow_name,
        "n_samples": n,
        "summary_path": str(summary_path),
    }

    # Compute mean, std, SE, and 95% CI for each dimension
    for col in dimension_score_columns:
        scores = summary_df[col].dropna().values
        mean = scores.mean()
        std = scores.std(ddof=1)
        se = std / np.sqrt(len(scores))
        ci_lower, ci_upper = bootstrap_ci(scores)

        row[f"{col}_mean"] = mean
        row[f"{col}_std"] = std
        row[f"{col}_se"] = se
        row[f"{col}_ci_lower"] = ci_lower
        row[f"{col}_ci_upper"] = ci_upper

    # Overall mean score
    overall_scores = summary_df["mean_score"].dropna().values
    row["mean_score"] = overall_scores.mean()
    row["mean_score_std"] = overall_scores.std(ddof=1)
    row["mean_score_se"] = overall_scores.std(ddof=1) / np.sqrt(len(overall_scores))
    ci_l, ci_u = bootstrap_ci(overall_scores)
    row["mean_score_ci_lower"] = ci_l
    row["mean_score_ci_upper"] = ci_u

    comparison_rows.append(row)

if not comparison_rows:
    raise ValueError("No comparison summaries were found.")

comparison_table = pd.DataFrame(comparison_rows)
comparison_table["reflection_label"] = comparison_table["source_reflection"].map(
    {False: "False", True: "True"}
)
comparison_table = comparison_table.sort_values(
    ["source_reflection", "workflow"]
).reset_index(drop=True)


# ── Step 2: Display summary table with mean and 95% CI ───────────────────────
print("\n=== MEAN SCORES WITH 95% BOOTSTRAP CONFIDENCE INTERVALS ===\n")

display_rows = []
for _, row in comparison_table.iterrows():
    display_row = {
        "source_reflection": row["reflection_label"],
        "workflow": row["workflow"],
        "n_samples": int(row["n_samples"]),
    }
    for col in dimension_score_columns:
        mean = row[f"{col}_mean"]
        lo = row[f"{col}_ci_lower"]
        hi = row[f"{col}_ci_upper"]
        display_row[col] = f"{mean:.3f} [{lo:.3f}, {hi:.3f}]"

    mean = row["mean_score"]
    lo = row["mean_score_ci_lower"]
    hi = row["mean_score_ci_upper"]
    display_row["mean_score"] = f"{mean:.3f} [{lo:.3f}, {hi:.3f}]"
    display_rows.append(display_row)

display(pd.DataFrame(display_rows))


# ── Step 3: Paired t-tests between configurations ────────────────────────────
print("\n=== PAIRED T-TESTS (14 months as paired observations) ===\n")

# Define the comparisons you care about
test_pairs = [
    ((False, "e2e"),     (False, "default"), "e2e vs default (no reflection)"),
    ((True,  "e2e"),     (True,  "default"), "e2e vs default (with reflection)"),
    ((True,  "default"), (False, "default"), "default: reflection vs no reflection"),
    ((True,  "e2e"),     (False, "e2e"),     "e2e: reflection vs no reflection"),
    ((False, "e2e"),     (True,  "default"), "best e2e vs best default"),
]

ttest_rows = []
for config_a, config_b, label in test_pairs:
    if config_a not in config_scores or config_b not in config_scores:
        print(f"Skipping {label} — data not available.")
        continue

    df_a = config_scores[config_a]
    df_b = config_scores[config_b]

    # Align on sample index for paired comparison
    shared_idx = df_a.index.intersection(df_b.index)
    if len(shared_idx) < 3:
        print(f"Skipping {label} — fewer than 3 shared samples.")
        continue

    for col in dimension_score_columns + ["mean_score"]:
        a = df_a.loc[shared_idx, col].values
        b = df_b.loc[shared_idx, col].values
        diff = a - b
        t_stat, p_value = stats.ttest_rel(a, b)

        ttest_rows.append({
            "comparison": label,
            "dimension": col,
            "mean_A": a.mean(),
            "mean_B": b.mean(),
            "mean_diff (A-B)": diff.mean(),
            "std_diff": diff.std(ddof=1),
            "t_stat": round(t_stat, 3),
            "p_value": round(p_value, 4),
            "significant_p05": p_value < 0.05,
            "significant_p10": p_value < 0.10,
        })

ttest_df = pd.DataFrame(ttest_rows)
if not ttest_df.empty:
    display(ttest_df.round(4))


# ── Step 4: Save all outputs ──────────────────────────────────────────────────
summary_save_path = comparison_root / "gpt-5_comparison_with_variance.csv"
comparison_table.to_csv(summary_save_path, index=False)
print(f"\nSaved full comparison table: {summary_save_path}")

ttest_save_path = comparison_root / "gpt-5_paired_ttests.csv"
if not ttest_df.empty:
    ttest_df.to_csv(ttest_save_path, index=False)
    print(f"Saved paired t-test results: {ttest_save_path}")


=== MEAN SCORES WITH 95% BOOTSTRAP CONFIDENCE INTERVALS ===



,source_reflection,workflow,n_samples,no_omissions_score,no_additions_score,grammaticality_score,coherence_score,fluency_score,mean_score
0,False,default,14,"4.643 [4.357, 4.857]","5.000 [5.000, 5.000]","4.929 [4.786, 5.000]","5.000 [5.000, 5.000]","5.000 [5.000, 5.000]","4.914 [4.843, 4.971]"
1,False,e2e,14,"4.571 [4.286, 4.857]","4.857 [4.643, 5.000]","4.929 [4.786, 5.000]","5.000 [5.000, 5.000]","5.000 [5.000, 5.000]","4.871 [4.771, 4.943]"
2,True,default,14,"4.643 [4.357, 4.857]","5.000 [5.000, 5.000]","4.929 [4.786, 5.000]","4.929 [4.786, 5.000]","5.000 [5.000, 5.000]","4.900 [4.829, 4.957]"
3,True,e2e,14,"4.643 [4.357, 4.857]","4.857 [4.643, 5.000]","5.000 [5.000, 5.000]","4.929 [4.786, 5.000]","5.000 [5.000, 5.000]","4.886 [4.800, 4.957]"
4,True,no_guardrail_no_finalizer,14,"4.643 [4.357, 4.857]","5.000 [5.000, 5.000]","5.000 [5.000, 5.000]","5.000 [5.000, 5.000]","4.929 [4.786, 5.000]","4.914 [4.843, 4.971]"
5,True,no_orchestrator_no_finalizer,14,"4.714 [4.500, 4.929]","4.071 [4.000, 4.214]","5.000 [5.000, 5.000]","4.929 [4.786, 5.000]","5.000 [5.000, 5.000]","4.743 [4.686, 4.800]"
6,True,no_orchestrator_no_guardrail_no_finalizer,14,"4.857 [4.643, 5.000]","5.000 [5.000, 5.000]","4.929 [4.786, 5.000]","5.000 [5.000, 5.000]","5.000 [5.000, 5.000]","4.957 [4.886, 5.000]"



=== PAIRED T-TESTS (14 months as paired observations) ===



,comparison,dimension,mean_A,mean_B,mean_diff (A-B),std_diff,t_stat,p_value,significant_p05,significant_p10
0,e2e vs default (no reflection),no_omissions_score,4.5714,4.6429,-0.0714,0.6157,-0.434,0.6714,False,False
1,e2e vs default (no reflection),no_additions_score,4.8571,5.0000,-0.1429,0.3631,-1.472,0.1648,False,False
2,e2e vs default (no reflection),grammaticality_score,4.9286,4.9286,0.0000,0.0000,NaN,NaN,False,False
3,e2e vs default (no reflection),coherence_score,5.0000,5.0000,0.0000,0.0000,NaN,NaN,False,False
4,e2e vs default (no reflection),fluency_score,5.0000,5.0000,0.0000,0.0000,NaN,NaN,False,False
5,e2e vs default (no reflection),mean_score,4.8714,4.9143,-0.0429,0.1399,-1.147,0.2722,False,False
6,e2e vs default (with reflection),no_omissions_score,4.6429,4.6429,0.0000,0.6794,0.000,1.0000,False,False
7,e2e vs default (with reflection),no_additions_score,4.8571,5.0000,-0.1429,0.3631,-1.472,0.1648,False,False
8,e2e vs default (with reflection),grammaticality_score,5.0000,4.9286,0.0714,0.2673,1.000,0.3356,False,False
9,e2e vs default (with reflection),coherence_score,4.9286,4.9286,0.0000,0.3922,0.000,1.0000,False,False



Saved full comparison table: /home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/validation/llm_judge/us/gpt-5/gpt-5_comparison_with_variance.csv
Saved paired t-test results: /home/chinonso/PHD_PROJECTS/Financial-D2T-Agent/results/validation/llm_judge/us/gpt-5/gpt-5_paired_ttests.csv
